# 🔍 Анализ рынка аналитических вакансий (HH.ru)

**Автор:** Виктор Кобцев  
**Дата создания:** 13.06.2026  
**Версия:** 1.0

---

## 📋 Описание проекта

Автономный ETL-пайплайн для мониторинга вакансий на HH.ru.  
Парсер ежедневно собирает данные, сохраняет исторические "слепки" в локальную базу DuckDB,  
на основе которых строится аналитический дашборд в Apache Superset.

**Цели проекта:**
- Анализ динамики рынка аналитических вакансий
- Мониторинг востребованных навыков и технологий
- Исследование зарплатных диапазонов по регионам и уровням

---

## 🗂️ Структура проекта

```
hh_analitics/
├── data/          # База данных DuckDB
├── notebooks/     # Jupyter ноутбуки
│   └── 01_parser.ipynb
└── logs/          # Логи парсера
```

## 1. Импорт библиотек

In [130]:
import re
import pandas as pd
import duckdb
from datetime import datetime
import time
import json
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

print("Все библиотеки загружены успешно!")
print(f"Pandas версия: {pd.__version__}")
print(f"DuckDB версия: {duckdb.__version__}")
print(f"Текущее время: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Все библиотеки загружены успешно!
Pandas версия: 2.3.3
DuckDB версия: 1.4.4
Текущее время: 2026-06-14 23:49:58


## 2. Настройка параметров

In [131]:
# Параметры поиска вакансий
CONFIG = {
    "keywords": "аналитик OR analyst OR analytics",
    "area": "113",
    "pages": 50,
}

# Базовый URL для поиска
BASE_URL = "https://hh.ru/search/vacancy"

# Пути
DB_PATH = "../data/hh_vacancies.duckdb"
LOG_PATH = "../logs/parser.log"

print("Конфигурация загружена:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print(f"  DB: {DB_PATH}")

Конфигурация загружена:
  keywords: аналитик OR analyst OR analytics
  area: 113
  pages: 50
  DB: ../data/hh_vacancies.duckdb


## 3. Парсинг вакансий с HH.ru

In [132]:
def create_driver():
    """Создаёт и настраивает браузер Chrome"""
    
    options = Options()
    options.add_argument("--headless")           # без графического интерфейса
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=ru-RU")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
    
    driver = webdriver.Chrome(options=options)
    print("Браузер запущен в фоновом режиме")
    return driver

# Проверяем
driver = create_driver()
print(f"Статус: OK")
driver.quit()
print("Браузер закрыт")

Браузер запущен в фоновом режиме
Статус: OK
Браузер закрыт


In [133]:
def get_salary(card):
    """Извлекает сырой текст зарплаты"""
    spans = card.find_all("span")
    for span in spans:
        text = span.get_text(strip=True)
        if "₽" in text and len(text) < 60 and len(span.find_all("span")) == 0:
            return text
    return None

def parse_salary(salary_raw):
    """Парсит зарплату: возвращает salary_from, salary_to, salary_currency"""
    if not salary_raw:
        return None, None, None
    
    # Убираем пробелы между цифрами (150 000 -> 150000)
    clean = re.sub(r'\s+', ' ', salary_raw)
    numbers = re.findall(r'\d[\d\s]*\d|\d+', clean)
    numbers = [int(n.replace(' ', '')) for n in numbers]
    
    # Определяем валюту
    currency = "RUB" if "₽" in salary_raw else None
    
    # Определяем от/до
    salary_from, salary_to = None, None
    
    if "от" in salary_raw.lower() and len(numbers) >= 1:
        salary_from = numbers[0]
    elif "до" in salary_raw.lower() and len(numbers) >= 1:
        salary_to = numbers[0]
    elif len(numbers) >= 2:
        salary_from = numbers[0]
        salary_to = numbers[1]
    elif len(numbers) == 1:
        salary_from = numbers[0]
        salary_to = numbers[0]
    
    return salary_from, salary_to, currency

def parse_vacancy_card(card):
    """Извлекает данные из одной карточки вакансии"""
    try:
        # Название
        title = card.find("a", {"data-qa": "serp-item__title"})
        title = title.text.strip() if title else None

        # Ссылка
        link = card.find("a", {"data-qa": "serp-item__title"})
        link = link.get("href") if link else None

        # Компания
        company = card.find(attrs={"data-qa": "vacancy-serp__vacancy-employer"})
        company = company.text.strip() if company else None

        # Город
        city = card.find(attrs={"data-qa": "vacancy-serp__vacancy-address"})
        city = city.text.strip() if city else None

        # Опыт
        exp = card.find(attrs={"data-qa": lambda x: x and "work-experience" in x})
        exp = exp.text.strip() if exp else None

        # Удалёнка
        remote = card.find(attrs={"data-qa": lambda x: x and "work-schedule-remote" in x})
        is_remote = True if remote else False

        # Отклик
        responded = card.find(attrs={"data-qa": "vacancy-serp__vacancy_responded"})
        is_applied = True if responded else False

        # Зарплата
        salary_raw = get_salary(card)
        salary_from, salary_to, salary_currency = parse_salary(salary_raw)

        return {
            "title": title,
            "company": company,
            "city": city,
            "experience": exp,
            "is_remote": is_remote,
            "is_applied": is_applied,
            "salary_raw": salary_raw,
            "salary_from": salary_from,
            "salary_to": salary_to,
            "salary_currency": salary_currency,
            "url": link,
            "snapshot_date": datetime.now().strftime("%Y-%m-%d"),
        }

    except Exception as e:
        print(f"Ошибка парсинга карточки: {e}")
        return None

print("Функции парсинга обновлены!")

Функции парсинга обновлены!


In [134]:
def fetch_vacancies(config):
    """Собирает вакансии по поисковому запросу"""
    
    driver = create_driver()
    all_vacancies = []
    
    try:
        for page in range(config["pages"]):
            url = (
                f"{BASE_URL}"
                f"?text={config['keywords']}"
                f"&area={config['area']}"
                f"&search_field=name"
                f"&page={page}"
            )
            
            driver.get(url)
            time.sleep(2)
            
            soup = BeautifulSoup(driver.page_source, "lxml")
            cards = soup.find_all("div", {"data-qa": "vacancy-serp__vacancy"})
            
            if not cards:
                print(f"Страница {page + 1}: вакансии не найдены, останавливаемся")
                break
            
            for card in cards:
                vacancy = parse_vacancy_card(card)
                if vacancy:
                    all_vacancies.append(vacancy)
            
            print(f"Страница {page + 1}: собрано {len(cards)} вакансий")
            time.sleep(1)
    
    finally:
        driver.quit()
    
    print(f"\nИтого собрано: {len(all_vacancies)} вакансий")
    return all_vacancies

In [135]:
# # Смотрим на данные
# df = pd.DataFrame(raw_vacancies)
# print(f"Размер датафрейма: {df.shape}")
# print(f"\nКолонки: {list(df.columns)}")
# print(f"\nПервые 3 строки:")
# df.head(3)

In [136]:
# df.describe()

## 4. Сохранение в базу данных

In [137]:
def save_to_db(df, db_path):
    con = duckdb.connect(db_path)
    con.execute("DROP TABLE IF EXISTS vacancies")  # Удаление базы
    
    df = df.drop_duplicates(subset=['url'])
    
    con.execute("""
        CREATE TABLE IF NOT EXISTS vacancies (
            title VARCHAR,
            company VARCHAR,
            city VARCHAR,
            experience VARCHAR,
            is_remote BOOLEAN,
            is_applied BOOLEAN,
            salary_raw VARCHAR,
            salary_from DOUBLE,
            salary_to DOUBLE,
            salary_currency VARCHAR,
            url VARCHAR,
            snapshot_date DATE,
            date_last_seen DATE
        )
    """)
    
    # Новые вакансии — которых ещё нет в базе
    con.execute("""
        INSERT INTO vacancies 
        SELECT *, snapshot_date as date_last_seen FROM df
        WHERE url NOT IN (SELECT url FROM vacancies)
    """)
    
    # Обновляем date_last_seen для уже существующих
    con.execute("""
        UPDATE vacancies
        SET date_last_seen = CURRENT_DATE
        WHERE url IN (SELECT url FROM df)
    """)
    
    count = con.execute("SELECT COUNT(*) FROM vacancies").fetchone()[0]
    today = con.execute("SELECT COUNT(*) FROM vacancies WHERE date_last_seen = CURRENT_DATE").fetchone()[0]
    print(f"Всего записей в базе: {count}")
    print(f"Активных сегодня: {today}")
    
    con.close()
    print("Данные сохранены!")

## 5. Запуск полного пайплайна

In [138]:
import logging

def setup_logger(log_path):
    """Настраивает логгер"""
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[
            logging.FileHandler(log_path, encoding="utf-8"),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

def run_pipeline():
    """Главная функция — запускает весь ETL пайплайн"""
    logger = setup_logger(LOG_PATH)
    logger.info("=== Запуск пайплайна ===")
    
    try:
        # Extract
        logger.info("Шаг 1: Парсинг вакансий...")
        raw_vacancies = fetch_vacancies(CONFIG)
        logger.info(f"Собрано вакансий: {len(raw_vacancies)}")
        
        # Transform
        logger.info("Шаг 2: Формирование датафрейма...")
        df = pd.DataFrame(raw_vacancies)
        df = df.dropna(subset=["title"])  # убираем пустые записи
        logger.info(f"Датафрейм: {df.shape[0]} строк, {df.shape[1]} колонок")
        
        # Load
        logger.info("Шаг 3: Сохранение в базу данных...")
        save_to_db(df, DB_PATH)
        
        logger.info("=== Пайплайн завершён успешно ===")
        return df
    
    except Exception as e:
        logger.error(f"Ошибка пайплайна: {e}")
        raise


In [139]:
# # Очищаем базу для чистого старта
# con = duckdb.connect(DB_PATH)
# con.execute("DELETE FROM vacancies")
# count = con.execute("SELECT COUNT(*) FROM vacancies").fetchone()[0]
# print(f"Записей в базе после очистки: {count}")
# con.close()

In [140]:
# Ручной запуск: только парсинг (без записи в БД)
logger = setup_logger(LOG_PATH)
raw_vacancies = fetch_vacancies(CONFIG)
df = pd.DataFrame(raw_vacancies)
df = df.dropna(subset=["title"])
logger.info(f"Датафрейм готов: {df.shape[0]} строк")

Браузер запущен в фоновом режиме
Страница 1: собрано 50 вакансий
Страница 2: собрано 50 вакансий
Страница 3: собрано 50 вакансий
Страница 4: собрано 50 вакансий
Страница 5: собрано 50 вакансий
Страница 6: собрано 50 вакансий
Страница 7: собрано 50 вакансий
Страница 8: собрано 50 вакансий
Страница 9: собрано 50 вакансий
Страница 10: собрано 50 вакансий
Страница 11: собрано 50 вакансий
Страница 12: собрано 50 вакансий
Страница 13: собрано 50 вакансий
Страница 14: собрано 50 вакансий
Страница 15: собрано 50 вакансий
Страница 16: собрано 50 вакансий
Страница 17: собрано 50 вакансий
Страница 18: собрано 50 вакансий
Страница 19: собрано 50 вакансий
Страница 20: собрано 50 вакансий
Страница 21: собрано 50 вакансий
Страница 22: собрано 50 вакансий
Страница 23: собрано 50 вакансий
Страница 24: собрано 50 вакансий
Страница 25: собрано 50 вакансий
Страница 26: собрано 50 вакансий
Страница 27: собрано 50 вакансий
Страница 28: собрано 50 вакансий
Страница 29: собрано 50 вакансий
Страница 30: собран

2026-06-14 23:55:01,162 | INFO | Датафрейм готов: 2000 строк



Итого собрано: 2000 вакансий


In [141]:
# Ручной запуск: только запись в БД
save_to_db(df, DB_PATH)
logger.info("=== Данные сохранены ===")

2026-06-14 23:55:01,690 | INFO | === Данные сохранены ===


Всего записей в базе: 1996
Активных сегодня: 1996
Данные сохранены!


In [142]:
# list(df['title'])

In [143]:
INCLUDE_WORDS = ['аналитик', 'analyst', 'analytics', 'аналитика']
mask = df['title'].str.lower().str.contains('|'.join(INCLUDE_WORDS))
print(f"Содержит: {mask.sum()} ({mask.sum()/len(df)*100:.1f}%)")
print(f"Не содержит: {(~mask).sum()} ({(~mask).sum()/len(df)*100:.1f}%)")

Содержит: 2000 (100.0%)
Не содержит: 0 (0.0%)


In [144]:
# keyword = "аналитик OR analyst OR analytics"
# area = "113"
# page = 0
# url = f"https://hh.ru/search/vacancy?text={keyword}&area={area}&page={page}"
# print(url)